In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!git clone -q https://github.com/tanyalebedeva/call-audio-analysis.git /kaggle/working/caa
%cd /kaggle/working/caa
!pip install -q -r requirements.txt
p = 'callaudio/audio.py'
s = open(p, encoding='utf-8').read()
s = s.replace("        wav = _load_with_ffmpeg(path, sr)\n", "        try:\n            wav = _load_with_ffmpeg(path, sr)\n        except Exception:\n            wav = np.zeros(int((duration or 1.0) * sr), dtype=np.float32)\n", 1)
open(p, 'w', encoding='utf-8').write(s)
!nvidia-smi -L

DATA, RUNS = '/tmp/data', '/kaggle/working/runs'
def run(cmd):
    print('\n>>>', cmd, flush=True)
    get_ipython().system(f'PYTHONUNBUFFERED=1 {cmd}')

run(f'python -m callaudio.data.prepare dusha --out {DATA} --work /tmp/work --per-class 6000 --test-per-class 1000')
run(f'python -m callaudio.data.prepare commands --out {DATA} --work /tmp/work --words core')

S = f'data_root={DATA} runs_root={RUNS} log_every=100'
for cfg, sets, mins in [
    ('configs/emotion/mel_cnn.yaml',       'epochs=6', 20),
    ('configs/vae/vae_mse.yaml',           'epochs=10', 10),
    ('configs/vae/cvae.yaml',              'epochs=10', 10),
    ('configs/emotion/hubert_frozen.yaml', 'epochs=2', 25),
    ('configs/emotion/mel_cnn.yaml',       f'epochs=6 loss.name=focal out_dir={RUNS}/emo_mel_focal', 15),
]:
    run(f'python -m callaudio.train --config {cfg} --set {S} {sets} max_minutes={mins}')

for r in ['emo_mel_ce', 'emo_hubert_frozen_ce', 'emo_mel_focal']:
    run(f'python -m callaudio.evaluate emotion --ckpt {RUNS}/{r}/best.pt')
for r in ['vae_mse', 'cvae_mse']:
    run(f'python -m callaudio.evaluate vae --ckpt {RUNS}/{r}/best.pt --pair yes,no --vocoder griffin_lim')

import json, glob
for f in sorted(glob.glob(f'{RUNS}/*/results_*.json')):
    r = json.load(open(f))
    keep = {k: r[k] for k in ['ua', 'wa', 'macro_f1', 'per_class_f1', 'best_layer', 'recon_mse_logmel',
            'latent_linear_probe_acc', 'latent_silhouette', 'judge_acc_real', 'judge_acc_recon',
            'judge_acc_generated'] if k in r}
    print(f.split('/')[-2], keep)
for f in sorted(glob.glob(f'{RUNS}/*/metrics.jsonl')):
    print(f.split('/')[-2], 'эпох:', sum(1 for _ in open(f)))
print('ГОТОВО')

/kaggle/working/caa
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
GPU 0: Tesla T4 (UUID: GPU-49ca6f5d-bd5a-85f8-c463-e95d6d9ae397)
GPU 1: Tesla T4 (UUID: GPU-bc20ef7f-c588-3c9c-9a18-556db102a569)

>>> python -m callaudio.data.prepare dusha --out /tmp/data --work /tmp/work --per-class 6000 --test-per-class 1000
Resolving data files: 100%|███████████████████| 48/48 [00:00<00:00, 8096.79it/s]
  train: {'angry': 427, 'neutral': 1026, 'positive': 327, 'sad': 220}
  train: {'angry': 857, 'neutral': 2064, 'positive': 664, 'sad': 415}
  train: {'angry': 1286, 'neutral': 3106, 'positive': 1003, 'sad': 605}
 

In [3]:
import os
os.chdir('/kaggle/working/caa')
os.system("sed -i 's#^data_root:.*#data_root: /tmp/data#; s#^runs_root:.*#runs_root: /kaggle/working/runs#' configs/paths.yaml")
print(open('configs/paths.yaml').read().splitlines()[1:3])

R = '/kaggle/working/runs'
T = 'python -m callaudio.train --config'
E = 'python -m callaudio.evaluate'

emo = f"""export PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0
{T} configs/emotion/mel_cnn.yaml --set log_every=100 epochs=6 max_minutes=20
{T} configs/emotion/hubert_frozen.yaml --set log_every=100 epochs=2 max_minutes=25
{T} configs/emotion/mel_cnn.yaml --set log_every=100 epochs=4 loss.name=focal out_dir={R}/emo_mel_focal max_minutes=12
{E} emotion --ckpt {R}/emo_mel_ce/best.pt
{E} emotion --ckpt {R}/emo_hubert_frozen_ce/best.pt
{E} emotion --ckpt {R}/emo_mel_focal/best.pt
echo ГОТОВО
"""

vae = f"""export PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=1
{T} configs/vae/vae_mse.yaml --set log_every=100 epochs=15 max_minutes=15
{T} configs/vae/cvae.yaml --set log_every=100 epochs=15 max_minutes=15
{E} vae --ckpt {R}/vae_mse/best.pt --pair yes,no --vocoder griffin_lim
{E} vae --ckpt {R}/cvae_mse/best.pt --pair yes,no --vocoder griffin_lim
echo ГОТОВО
"""

open('emo.sh', 'w').write(emo)
open('vae.sh', 'w').write(vae)
os.system('nohup bash emo.sh > emo.log 2>&1 &')
os.system('nohup bash vae.sh > vae.log 2>&1 &')
print('запущено')

['data_root: /tmp/data', 'runs_root: /kaggle/working/runs']
запущено


In [8]:
!tail -n 4 /kaggle/working/caa/emo.log
!echo -----
!tail -n 4 /kaggle/working/caa/vae.log

  эпоха 2 шаг 200/337 loss=1.0532 acc=0.5521
  эпоха 2 шаг 300/337 loss=1.0331 acc=0.5639
[эпоха 2/6] train_loss=1.0316 train_acc=0.5656 val_ua=0.5533 val_wa=0.5533 val_macro_f1=0.4923 val_score=0.4923
  ↑ новый лучший score 0.5177, сохранён best.pt
-----
  ↑ новый лучший score -0.0918, сохранён best.pt
  эпоха 2 шаг 100/240 loss=506.2042 recon=470.5117 kl=148.4638 beta=0.2412
  эпоха 2 шаг 200/240 loss=497.7634 recon=458.8830 kl=139.1684 beta=0.2829
[эпоха 2/15] train_loss=496.2645 train_recon=456.0847 train_kl=136.2991 train_beta=0.2996 val_recon_mse=0.0958 val_kl=126.4674 val_score=-0.0958


In [9]:
import os
os.chdir('/kaggle/working/caa')
os.system('pkill -f "bash emo.sh"; pkill -f "configs/emotion"; sleep 3; rm -rf /kaggle/working/runs/emo_mel_ce')

R = '/kaggle/working/runs'
T = 'python -m callaudio.train --config'
E = 'python -m callaudio.evaluate'

emo = f"""export PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0
{T} configs/emotion/mel_cnn.yaml --set log_every=100 epochs=15 max_minutes=30
{T} configs/emotion/hubert_frozen.yaml --set log_every=100 epochs=4 max_minutes=25
{T} configs/emotion/mel_cnn.yaml --set log_every=100 epochs=8 loss.name=focal out_dir={R}/emo_mel_focal max_minutes=16
{E} emotion --ckpt {R}/emo_mel_ce/best.pt
{E} emotion --ckpt {R}/emo_hubert_frozen_ce/best.pt
{E} emotion --ckpt {R}/emo_mel_focal/best.pt
echo ГОТОВО
"""

sep = f"""export PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=1
python -m callaudio.data.prepare golos --out /tmp/data --work /tmp/work
python -m callaudio.data.prepare musan --out /tmp/data --work /tmp/work --music-files 300
rm -f /tmp/work/*.tar.gz /tmp/work/*.tar
{T} configs/separation/unet_sisdr.yaml --set log_every=50 epochs=25 max_minutes=55
{E} separation --ckpt {R}/sep_unet_sisdr/best.pt --limit 200 --wer --whisper small
echo ГОТОВО
"""

open('emo.sh', 'w').write(emo)
open('sep.sh', 'w').write(sep)
os.system('nohup bash emo.sh > emo.log 2>&1 &')
os.system('nohup bash sep.sh > sep.log 2>&1 &')
print('запущено')

запущено


In [72]:
for f in ['emo', 'vae', 'sep', 'hub', 'extra']:
    print(f'===== {f}')
    !tail -n 4 /kaggle/working/caa/{f}.log

===== emo
                     ~~~~^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/runs/emo_hubert_frozen_ce/best.pt'
[evaluate] результаты: /kaggle/working/runs/emo_mel_focal/results_emotion.json
ГОТОВО
===== vae
[train] готово: {'best_score': -0.06833514193731355, 'out_dir': '/kaggle/working/runs/cvae_mse'}
[evaluate] результаты: /kaggle/working/runs/vae_mse/results_vae.json
[evaluate] результаты: /kaggle/working/runs/cvae_mse/results_vae.json
ГОТОВО
===== sep
    return speech + noise, noise
           ~~~~~~~^~~~~~~
ValueError: operands could not be broadcast together with shapes (64320,) (64319,) 
ГОТОВО
===== hub
Loading weights: 100%|██████████| 211/211 [00:00<00:00, 11603.49it/s]
[evaluate] результаты: /kaggle/working/runs/emo_hubert_frozen_ce/results_emotion.json
ГОТОВО
===== extra
  ↑ новый лучший score -0.0627, сохранён best.pt
[train] готово: {'best_score': -0.06266145786014467, 'out_dir': '/kaggle/working/runs/cvae_mse'}
[evaluate] рез

In [22]:
!ps aux | grep "callaudio.train" | grep -v grep | awk '{print $2, $13, $14, $15}'

313 callaudio.train --config configs/vae/vae_mse.yaml
331 callaudio.train --config configs/vae/vae_mse.yaml
358 callaudio.train --config configs/vae/vae_mse.yaml
359 callaudio.train --config configs/vae/vae_mse.yaml
363 callaudio.train --config configs/vae/vae_mse.yaml
370 callaudio.train --config configs/vae/vae_mse.yaml
483 callaudio.train --config configs/emotion/mel_cnn.yaml
506 callaudio.train --config configs/emotion/mel_cnn.yaml
507 callaudio.train --config configs/emotion/mel_cnn.yaml


In [15]:
!pkill -f "mel_cnn.yaml --set log_every=100 epochs=6 "
!sleep 2; ps -eo pid,args | grep "callaudio.train" | grep -v grep | grep -o "^ *[0-9]* .*yaml.*epochs=[0-9]*" | cut -c1-120

    313 python3 -m callaudio.train --config configs/vae/vae_mse.yaml --set log_every=100 epochs=15
    331 python3 -m callaudio.train --config configs/vae/vae_mse.yaml --set log_every=100 epochs=15
    358 python3 -m callaudio.train --config configs/vae/vae_mse.yaml --set log_every=100 epochs=15
    359 python3 -m callaudio.train --config configs/vae/vae_mse.yaml --set log_every=100 epochs=15
    363 python3 -m callaudio.train --config configs/vae/vae_mse.yaml --set log_every=100 epochs=15
    370 python3 -m callaudio.train --config configs/vae/vae_mse.yaml --set log_every=100 epochs=15
    483 python3 -m callaudio.train --config configs/emotion/mel_cnn.yaml --set log_every=100 epochs=15
    506 python3 -m callaudio.train --config configs/emotion/mel_cnn.yaml --set log_every=100 epochs=15
    507 python3 -m callaudio.train --config configs/emotion/mel_cnn.yaml --set log_every=100 epochs=15


In [20]:
!pkill -f "mel_cnn.yaml --set log_every=100 epochs=6 "

In [27]:
import os
os.chdir('/kaggle/working/caa')
R = '/kaggle/working/runs'
extra = f"""export PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0
while pgrep -f "bash emo.sh" > /dev/null; do sleep 20; done
python -m callaudio.train --config configs/vae/cvae.yaml --set log_every=100 epochs=15 max_minutes=18
python -m callaudio.evaluate vae --ckpt {R}/cvae_mse/best.pt --pair yes,no --vocoder griffin_lim
echo ГОТОВО
"""
open('extra.sh', 'w').write(extra)
os.system('nohup bash extra.sh > extra.log 2>&1 &')
print('запущено')

запущено


In [30]:
!grep -v "^  [0-9]* / " /kaggle/working/caa/sep.log | tail -n 45


During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ~~~~~~~~~~~~~~~^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/collate.py", line 401, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/collate.py", line 192, in collate
    key: collate([d[key] for d in batch], collate_fn_map=collate_fn_map)
         ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/collate.py", line 243, in collate
    raise Typ

In [31]:
import os
os.chdir('/kaggle/working/caa')
os.system("""sed -i 's/row.get("text", "")/(row.get("text") or "")/' callaudio/data/datasets.py""")
os.system('grep -n "row.get(\\"text\\")" callaudio/data/datasets.py')

R = '/kaggle/working/runs'
sep = f"""export PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=1
rm -rf {R}/sep_unet_sisdr
python -m callaudio.train --config configs/separation/unet_sisdr.yaml --set log_every=50 epochs=25 max_minutes=50
python -m callaudio.evaluate separation --ckpt {R}/sep_unet_sisdr/best.pt --limit 150 --wer --whisper small
echo ГОТОВО
"""
open('sep.sh', 'w').write(sep)
os.system('nohup bash sep.sh > sep.log 2>&1 &')
print('запущено')

76:        return wav, (row.get("text") or "")
запущено


In [35]:
!grep -n -i "hubert\|error\|готово" /kaggle/working/caa/emo.log | cut -c1-200 | head -20

63:[train] готово: {'best_score': 0.7396351132652498, 'out_dir': '/kaggle/working/runs/emo_mel_ce'}
91:RuntimeError: The size of tensor a (13) must match the size of tensor b (11) at non-singleton dimension 0


In [37]:
import os
os.chdir('/kaggle/working/caa')
os.system("""sed -i 's/AutoModel.from_pretrained(pretrained)/AutoModel.from_pretrained(pretrained, layerdrop=0.0)/' callaudio/models/emotion.py""")
os.system("""sed -i 's/w = torch.softmax(self.layer_weights, dim=0)/w = torch.softmax(self.layer_weights[: hs.shape[0]], dim=0)/' callaudio/models/emotion.py""")
os.system('grep -n "layerdrop\\|hs.shape" callaudio/models/emotion.py')

R = '/kaggle/working/runs'
hub = f"""export PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0
rm -rf {R}/emo_hubert_frozen_ce
python -m callaudio.train --config configs/emotion/hubert_frozen.yaml --set log_every=100 epochs=4 max_minutes=30
python -m callaudio.evaluate emotion --ckpt {R}/emo_hubert_frozen_ce/best.pt
echo ГОТОВО
"""
open('hub.sh', 'w').write(hub)
os.system('nohup bash hub.sh > hub.log 2>&1 &')
print('запущено')

146:        self.backbone = AutoModel.from_pretrained(pretrained, layerdrop=0.0)
171:        w = torch.softmax(self.layer_weights[: hs.shape[0]], dim=0)[:, None, None, None]
запущено


In [39]:
!tail -n 4 /kaggle/working/caa/hub.log

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 6822.85it/s]
[train] emotion · SSLClassifier · параметров 94.96 M (обучаемых 0.59 M) · устройство cuda


In [71]:
import glob
for f in sorted(glob.glob('/kaggle/working/runs/*/results_*.json')):
    print('==', f)
    print(open(f, encoding='utf-8').read()[:2500])
!grep -h "\[эпоха\|готово" /kaggle/working/caa/emo.log /kaggle/working/caa/vae.log

== /kaggle/working/runs/cvae_mse/results_vae.json
{
  "checkpoint": "/kaggle/working/runs/cvae_mse/best.pt",
  "words": [
    "down",
    "go",
    "left",
    "no",
    "off",
    "on",
    "right",
    "stop",
    "up",
    "yes"
  ],
  "conditional": true,
  "recon_mse_logmel": 0.4565991759300232,
  "latent_linear_probe_acc": 0.27,
  "latent_silhouette": -0.02756311744451523,
  "judge_acc_real": 0.491,
  "judge_acc_recon": 0.647,
  "judge_acc_generated": 0.4,
  "interpolation": {
    "from": "yes",
    "to": "no",
    "p_from": [
      0.776,
      0.878,
      0.744,
      0.405,
      0.001,
      0.0,
      0.0,
      0.0
    ],
    "p_to": [
      0.002,
      0.013,
      0.047,
      0.123,
      0.372,
      0.432,
      0.531,
      0.68
    ]
  }
}
== /kaggle/working/runs/emo_hubert_frozen_ce/results_emotion.json
{
  "ua": 0.7977500000000001,
  "wa": 0.79775,
  "macro_f1": 0.7996936075259248,
  "per_class_f1": [
    0.6972563859981078,
    0.8355464759959141,
    0.83204930

In [67]:
import os
os.chdir('/kaggle/working/caa')
s = open('callaudio/audio.py', encoding='utf-8').read()
s = s.replace("    noise = noise * gain\n", "    noise = np.resize(noise, len(speech)) * gain\n")
open('callaudio/audio.py', 'w', encoding='utf-8').write(s)
os.system('grep -n "np.resize" callaudio/audio.py')

R = '/kaggle/working/runs'
ev = f"""export PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=1
python -m callaudio.evaluate separation --ckpt {R}/sep_unet_sisdr/best.pt --limit 150 --wer --whisper small
echo ГОТОВО
"""
open('ev.sh', 'w').write(ev)
os.system('nohup bash ev.sh > ev.log 2>&1 &')
print('запущено')

92:    noise = np.resize(noise, len(speech)) * gain
запущено


In [70]:
!tail -n 3 /kaggle/working/caa/ev.log

  150/150
[evaluate] результаты: /kaggle/working/runs/sep_unet_sisdr/results_separation.json
ГОТОВО


In [73]:
import os, glob, json
os.chdir('/kaggle/working/caa')
from IPython.display import Audio, display
from callaudio.pipeline import CallAnalyzer
from callaudio.audio import load_audio

R = '/kaggle/working/runs'
an = CallAnalyzer(enhancer_ckpt=f'{R}/sep_unet_sisdr/best.pt',
                  emotion_ckpt=f'{R}/emo_mel_ce/best.pt', whisper_size='small')

print('ОЧИСТКА + РАСШИФРОВКА + ЭМОЦИЯ')
for mix in sorted(glob.glob(f'{R}/sep_unet_sisdr/examples/*_mix.wav'))[:4]:
    print('\n', os.path.basename(mix))
    print('с шумом:');   display(Audio(mix))
    print('очищено:');   display(Audio(mix.replace('_mix', '_enhanced')))
    print('оригинал:');  display(Audio(mix.replace('_mix', '_clean')))
    r = an.analyze(mix)
    for s in r['segments']:
        print(f"  [{s['start']}–{s['end']} с] {s['text']}  |  {s['emotion']} {s['confidence']}")

print('\nЭМОЦИИ НА ТЕСТЕ DUSHA')
rows = [json.loads(l) for l in open('/tmp/data/manifests/emotion_test.jsonl', encoding='utf-8')]
picked = {}
for row in rows:
    picked.setdefault(row['label'], row)
for label, row in picked.items():
    wav = load_audio(row['path'])
    probs = an.emotion(wav)
    pred = max(probs, key=probs.get)
    print(f'\nнастоящая: {label}  |  модель: {pred} ({probs[pred]:.2f})')
    display(Audio(row['path']))

ОЧИСТКА + РАСШИФРОВКА + ЭМОЦИЯ

 00_snr-5_mix.wav
с шумом:


очищено:


оригинал:


  [0.7–3.3 с] поставь белый альбом Burrito  |  positive 0.703

 01_snr0_mix.wav
с шумом:


очищено:


...

оригинал:


  [0.9–3.1 с] Строшилки 18+.  |  neutral 0.794

ЭМОЦИИ НА ТЕСТЕ DUSHA

настоящая: neutral  |  модель: sad (0.89)



настоящая: sad  |  модель: sad (0.57)



настоящая: angry  |  модель: angry (0.72)



настоящая: positive  |  модель: positive (0.47)


In [74]:
!cd /kaggle/working && cp caa/*.log runs/ && zip -qr results.zip runs -x "*/last.pt" "runs/emo_hubert_frozen_ce/*.pt" && ls -lh results.zip

-rw-r--r-- 1 root root 114M Oct 27 07:19 results.zip
